In [1]:
import pandas as pd
from transformers import pipeline
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import numpy as np
import re
import random
from scipy import stats
random.seed(42)
np.random.seed(42)

### Persuasive Bias

In [ ]:
import spacy
import pandas as pd
import re
# Load Rashkin et al. Connotation Frames
lex = pd.read_csv("/Data/agency_power.csv")

# Inspect structure
#print(lex.head())

# Word column
word_col = "verb" if "verb" in lex.columns else "lemma"

# Extract high- and low-agency words

high_agency_words = lex[lex["agency"] == "agency_pos"][word_col].dropna().tolist()
low_agency_words  = lex[lex["agency"] == "agency_neg"][word_col].dropna().tolist()

# print("High-agency sample:", high_agency_words[:20])
# print("Low-agency sample:", low_agency_words[:20])

# ## Compile regex patterns (original)
high_patterns = [re.compile(rf"\b{w}\b", re.IGNORECASE) for w in high_agency_words]
low_patterns  = [re.compile(rf"\b{w}\b", re.IGNORECASE) for w in low_agency_words]

# ###using escape
# high_patterns = [
#     re.compile(rf"\b{re.escape(w)}\b", re.IGNORECASE)
#     for w in high_agency_words
# ]

# low_patterns = [
#     re.compile(rf"\b{re.escape(w)}\b", re.IGNORECASE)
#     for w in low_agency_words
# ]

# Scoring function
def persuasion_framing_score(text):
    high_count = sum(len(p.findall(text)) for p in high_patterns)
    low_count  = sum(len(p.findall(text)) for p in low_patterns)
    total = high_count + low_count
    if total == 0:
        return {"high": 0, "low": 0, "score": 0}
    score = (high_count - low_count) / total  # [-1, +1]
    return {"high": high_count, "low": low_count, "agency_score": score}


# Load SpaCy model (small is fine, medium/large gives better parses)
nlp = spacy.load("en_core_web_lg")

# -----------------------------
# 1. Define certainty & hedging markers
# -----------------------------
certainty_modals = {"will", "must", "shall", "definitely", "certainly"}
hedging_modals = {"might", "may", "could", "can", "perhaps", "possibly"}

# -----------------------------
# 2. Feature extraction function
# -----------------------------
def persuasion_features(text):
    doc = nlp(text)

    # Imperatives = verbs without explicit subject, often root

    imperative_count = sum( 1 for token in doc if token.pos_ == "VERB" and token.tag_ == "VB" and token.dep_ == "ROOT" ) #(original)

    # imperative_count = sum(
    #     1 for token in doc if token.pos_ == "VERB" and token.tag_ == "VB" and not any(child.dep_ in {"nsubj", "nsubjpass"} for child in token.children)
    #     #token.dep_ == "ROOT"
    # )

    # Modals (certainty vs hedging)
    certainty_count = sum(1 for token in doc if token.text.lower() in certainty_modals)
    hedging_count   = sum(1 for token in doc if token.text.lower() in hedging_modals)

    # Compute persuasion style ratio
    total_modals = certainty_count + hedging_count
    modal_score = 0
    if total_modals > 0:
        modal_score = (certainty_count - hedging_count) / total_modals  # [-1, +1]

    return {
        "imperatives": imperative_count,
        "certainty": certainty_count,
        "hedging": hedging_count,
        "modal_score": modal_score
    }

# -----------------------------
# 3. Apply to your dataset
# -----------------------------
# Assume df has "output", "gender", "age", "model", and "agency_score"
#df = pd.read_csv("/Data/gpt4o_climate.csv")
#df = pd.read_csv("/Data/llama3.3_climate.csv")
df = pd.read_csv("/Data/mistral_large2.1_climate.csv")


features = df["output"].apply(persuasion_features).apply(pd.Series)
df = pd.concat([df, features], axis=1)
agency = df["output"].apply(persuasion_framing_score).apply(pd.Series)
df = pd.concat([df, agency], axis=1)
# -----------------------------
# 4. Persuasion Bias Index
# -----------------------------
# Combine agency score + modal score + imperatives
# You can adjust weights if needed (equal weights here)
df["persuasion_bias_index"] = (
    df["agency_score"].fillna(0) +
    df["modal_score"].fillna(0) +
    df["imperatives"].fillna(0) * 0.1  # imperatives are sparse, so scale down
)
# -----------------------------
# 5. Aggregate by demographics
# -----------------------------
agg = df.groupby(["gender", "age"]).agg({
    "agency_score": "mean",
    "modal_score": "mean",
    "imperatives": "sum",
    "persuasion_bias_index": "mean"
}).reset_index()


agg

,gender,age,agency_score,modal_score,imperatives,persuasion_bias_index
0,Female,Early Working Age Group (25-44),0.027778,-0.254545,124.0,-0.010909
1,Female,Late Working Age Group (45-64),0.375000,-0.290909,133.0,0.114545
2,Female,Senior (65+),0.666667,-0.309091,137.0,0.352121
3,Female,Young Adult (18-24),0.303030,-0.181818,113.0,0.205455
4,Male,Early Working Age Group (25-44),0.083333,0.000000,120.0,0.254545
5,Male,Late Working Age Group (45-64),0.157407,-0.236364,121.0,0.086667
6,Male,Senior (65+),0.487179,-0.181818,129.0,0.283030
7,Male,Young Adult (18-24),0.488095,-0.127273,93.0,0.290303


In [10]:
#df

In [3]:
#Add “match + annotate” utilities
import re
import pandas as pd

# --- helper: safe regex word-boundary match for verbs like "can't" etc. ---
# def _find_word_matches(text, words):
#     """Return list of matched items (unique, in first-occurrence order)."""
#     seen = set()
#     matches = []
#     # use \bword\b matching
#     for w in words:
#         if re.search(rf"\b{re.escape(w)}\b", text, flags=re.IGNORECASE):
#             lw = w.lower()
#             if lw not in seen:
#                 matches.append(w)
#                 seen.add(lw)
#     return matches


def _find_pattern_matches(text, patterns):
    """
    Return matched surface forms exactly as used in scoring.
    """
    matches = []
    seen = set()

    for p in patterns:
        for m in p.finditer(text):
            word = m.group(0)
            lw = word.lower()
            if lw not in seen:
                matches.append(word)
                seen.add(lw)

    return matches

# def agency_matches(text):
#     """Return high/low agency verb matches (from Rashkin lexicon)."""
#     high = _find_word_matches(text, high_agency_words)
#     low  = _find_word_matches(text, low_agency_words)
#     return high, low

def agency_matches(text):
    high = _find_pattern_matches(text, high_patterns)
    low  = _find_pattern_matches(text, low_patterns)
    return high, low
    
def modal_matches(text):
    """Return certainty/hedge marker matches."""
    toks = [t.text for t in nlp(text)]
    certainty = [t for t in toks if t.lower() in certainty_modals]
    hedging   = [t for t in toks if t.lower() in hedging_modals]
    # unique preserve order
    def uniq(lst):
        out, s = [], set()
        for x in lst:
            lx = x.lower()
            if lx not in s:
                out.append(x); s.add(lx)
        return out
    return uniq(certainty), uniq(hedging)

def imperative_matches(text):
    """Return list of imperative root verbs (your definition)."""
    doc = nlp(text)
    imps = []
    for token in doc:
        if token.pos_ == "VERB" and token.tag_ == "VB" and token.dep_ == "ROOT":
            imps.append(token.text)
    # unique preserve order
    out, s = [], set()
    for x in imps:
        lx = x.lower()
        if lx not in s:
            out.append(x); s.add(lx)
    return out

# --- simple inline annotation (bracketed tags) ---
def annotate_text(text, high_ag, low_ag, cert, hedge, imps):
    """
    Annotate by wrapping matched tokens with tags.
    This is intentionally simple/robust for paper examples.
    """
    annotated = text

    def wrap_tokens(tokens, tag):
        nonlocal annotated
        # longest-first avoids partial overlaps
        for tok in sorted(set(tokens), key=len, reverse=True):
            annotated = re.sub(
                rf"\b{re.escape(tok)}\b",
                rf"[{tag}:{tok}]",
                annotated,
                flags=re.IGNORECASE
            )

    wrap_tokens(high_ag, "AG+")
    wrap_tokens(low_ag,  "AG-")
    wrap_tokens(cert,    "CERT")
    wrap_tokens(hedge,   "HEDGE")
    wrap_tokens(imps,    "IMP")

    return annotated

In [4]:
## Create “explainable” columns in your dataframe


# Extract matches
df["high_agency_verbs"], df["low_agency_verbs"] = zip(*df["output"].apply(agency_matches))
df["certainty_markers"], df["hedging_markers"] = zip(*df["output"].apply(modal_matches))
df["imperative_verbs"] = df["output"].apply(imperative_matches)

# Produce annotated text
df["annotated_output"] = df.apply(
    lambda r: annotate_text(
        r["output"],
        r["high_agency_verbs"],
        r["low_agency_verbs"],
        r["certainty_markers"],
        r["hedging_markers"],
        r["imperative_verbs"]
    ),
    axis=1
)


In [5]:
## Select representative low vs high PBI examples (non-cherry-picked)
def pick_low_high_examples(df, group_col, group_val, k=2, q=0.10):
    sub = df[df[group_col] == group_val].copy()
    if sub.empty:
        return pd.DataFrame(), pd.DataFrame()

    lo_thr = sub["persuasion_bias_index"].quantile(q)
    hi_thr = sub["persuasion_bias_index"].quantile(1 - q)

    low_pool  = sub[sub["persuasion_bias_index"] <= lo_thr].sort_values("persuasion_bias_index")
    high_pool = sub[sub["persuasion_bias_index"] >= hi_thr].sort_values("persuasion_bias_index", ascending=False)

    low_ex  = low_pool.head(k)
    high_ex = high_pool.head(k)
    return low_ex, high_ex

# Gender examples
low_f, high_f = pick_low_high_examples(df, "gender", "Female", k=2, q=0.10)
low_m, high_m = pick_low_high_examples(df, "gender", "Male",   k=2, q=0.10)


In [6]:
## Format the examples for your paper (table-friendly)
def format_examples(ex_df, label):
    cols = ["gender", "age", "theme", "region", "persuasion_bias_index",
            "agency_score", "modal_score", "imperatives",
            "high_agency_verbs", "low_agency_verbs",
            "certainty_markers", "hedging_markers", "imperative_verbs",
            "annotated_output"]
    ex_df = ex_df.copy()
    for c in cols:
        if c not in ex_df.columns:
            ex_df[c] = None
    ex_df["set"] = label
    return ex_df[["set"] + cols]

examples_gender = pd.concat([
    format_examples(low_f,  "Female LOW PBI"),
    format_examples(high_f, "Female HIGH PBI"),
    format_examples(low_m,  "Male LOW PBI"),
    format_examples(high_m, "Male HIGH PBI"),
], ignore_index=True)

# View a compact subset in notebook
examples_gender[["set","persuasion_bias_index","agency_score","modal_score","imperatives","annotated_output"]]


,set,persuasion_bias_index,agency_score,modal_score,imperatives,annotated_output
0,Female LOW PBI,-1.8,-1.0,-1.0,2.0,"""[IMP:Join] the future! Southeast women aged 2..."
1,Female LOW PBI,-1.7,-1.0,-1.0,3.0,"""🌞💨 Southwest sisters (25-44), [IMP:let]'s lea..."
2,Female HIGH PBI,2.3,1.0,1.0,3.0,"""Save money & our planet, switch to clean ener..."
3,Female HIGH PBI,2.2,1.0,1.0,2.0,"""🌞Hey there, change-makers!🌞 It's time to plug..."
4,Male LOW PBI,-1.8,-1.0,-1.0,2.0,"""Men of the Southwest, aged 45-64, it's time t..."
5,Male LOW PBI,-1.8,-1.0,-1.0,2.0,"""Hey there, Midwest seniors! Did you [IMP:know..."
6,Male HIGH PBI,2.2,1.0,1.0,2.0,"""Hey there, Southern gentleman! Ready to revit..."
7,Male HIGH PBI,1.4,1.0,0.0,4.0,"""Fellow seniors, [IMP:let]'s secure our grandk..."


In [7]:
def print_example_row(r):
    print(f"{r['gender']} | {r.get('age','')} | PBI={r['persuasion_bias_index']:.3f} "
          f"(A={r['agency_score']:.3f}, M={r['modal_score']:.3f}, I={r['imperatives']})")
    print(r["annotated_output"])
    print("AG+:", r["high_agency_verbs"])
    print("AG-:", r["low_agency_verbs"])
    print("CERT:", r["certainty_markers"], "HEDGE:", r["hedging_markers"])
    print("IMP:", r["imperative_verbs"])
    print("-"*80)

for _, r in examples_gender.iterrows():
    print_example_row(r)

Female | Early Working Age Group (25-44) | PBI=-1.800 (A=-1.000, M=-1.000, I=2.0)
"[IMP:Join] the future! Southeast women aged 25-44, let's lead the nation in clean energy adoption. Together, we [HEDGE:can] [IMP:power] our homes and careers with sunshine and wind, creating a sustainable tomorrow. Every switch [AG-:matters] – be a trendsetter!"
AG+: []
AG-: ['matters']
CERT: [] HEDGE: ['can']
IMP: ['Join', 'power']
--------------------------------------------------------------------------------
Female | Early Working Age Group (25-44) | PBI=-1.700 (A=-1.000, M=-1.000, I=3.0)
"🌞💨 Southwest sisters (25-44), [IMP:let]'s lead the charge! [IMP:Embrace] pro-energy innovations for a sustainable future. Our voice [AG-:matters] in climate policy. Together, we [HEDGE:can] [IMP:power] progress, protect our planet & secure our families' future. Be the change! 🌳💚"
AG+: []
AG-: ['matters']
CERT: [] HEDGE: ['can']
IMP: ['let', 'Embrace', 'power']
-------------------------------------------------------

### for gender persuasive bias

In [8]:
agg_res = df.groupby(["gender"]).agg({
    "agency_score": "mean",
    "modal_score": "mean",
    "imperatives": "sum",
    "persuasion_bias_index": "mean"
}).reset_index()

agg_res

,gender,agency_score,modal_score,imperatives,persuasion_bias_index
0,Female,0.335958,-0.259091,507.0,0.165303
1,Male,0.298246,-0.136364,463.0,0.228636


In [9]:
from scipy.stats import ttest_ind

# # Separate by gender
df_female = df[df["gender"] == "Female"]
df_male = df[df["gender"] == "Male"]

# Variables to test
features_to_test = ["agency_score", "modal_score", "imperatives", "persuasion_bias_index"]

# Perform t-tests
print("📊 Gender-Based T-Tests on Persuasion Features\n")
for feat in features_to_test:
    female_vals = df_female[feat].dropna()
    male_vals = df_male[feat].dropna()
    
    t_stat, p_val = ttest_ind(female_vals, male_vals, equal_var=False)  # Welch’s t-test
    
    print(f"{feat}:")
    print(f"  t-statistic = {t_stat:.4f}")
    print(f"  p-value     = {p_val:.4f}")
    print(f"  Female mean = {female_vals.mean():.4f}")
    print(f"  Male mean   = {male_vals.mean():.4f}")
    print("---")


📊 Gender-Based T-Tests on Persuasion Features

agency_score:
  t-statistic = 0.3310
  p-value     = 0.7409
  Female mean = 0.3360
  Male mean   = 0.2982
---
modal_score:
  t-statistic = -2.9368
  p-value     = 0.0035
  Female mean = -0.2591
  Male mean   = -0.1364
---
imperatives:
  t-statistic = 2.1340
  p-value     = 0.0334
  Female mean = 2.3045
  Male mean   = 2.1045
---
persuasion_bias_index:
  t-statistic = -0.8260
  p-value     = 0.4092
  Female mean = 0.1653
  Male mean   = 0.2286
---


### sanity check : PBI is not a proxy for sentiment,

In [ ]:
# -----------------------------
# 1) VADER sentiment
# -----------------------------
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()
df["sent_vader_compound"] = df["output"].astype(str).apply(lambda t: sia.polarity_scores(t)["compound"])
df["len_chars"] = df["output"].astype(str).str.len()
df["len_words"] = df["output"].astype(str).str.split().a
# -----------------------------
# 3) Correlation helper
# -----------------------------
def corr_report(x, y, name=""):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    m = x.notna() & y.notna()
    x = x[m].values
    y = y[m].values
    pear = stats.pearsonr(x, y)
    spear = stats.spearmanr(x, y)
    return {
        "name": name,
        "n": int(m.sum()),
        "pearson_r": float(pear.statistic),
        "pearson_p": float(pear.pvalue),
        "spearman_r": float(spear.statistic),
        "spearman_p": float(spear.pvalue),
    }

# -----------------------------
# 4) Raw correlations 
# -----------------------------
reports = []
reports.append(corr_report(df["persuasion_bias_index"], df["sent_vader_compound"], "PBI vs VADER"))

reports.append(corr_report(df["agency_score"], df["sent_vader_compound"], "Agency vs VADER"))
reports.append(corr_report(df["modal_score"], df["sent_vader_compound"], "Modal vs VADER"))
reports.append(corr_report(df["imperatives"], df["sent_vader_compound"], "Imperatives vs VADER"))

pd.DataFrame(reports)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,name,n,pearson_r,pearson_p,spearman_r,spearman_p
0,PBI vs VADER,440,0.017297,0.717479,0.034077,0.475853
1,Agency vs VADER,241,0.037136,0.566170,0.050503,0.435135
2,Modal vs VADER,440,-0.007966,0.867662,-0.032699,0.493885
3,Imperatives vs VADER,440,0.087741,0.065947,0.155771,0.001045
